# FinSight AI · Research Part 3: A personal-ledger test set (English + Hinglish)

Parts 1 and 2 measured FinSight's building blocks on **corporate** finance data (Banking77, FiQA, FinQA, TAT-QA, ConvFinQA). The literature review found the main gap: **no benchmark tests an assistant on a person's own transactions with checked numbers, and none asks those questions in Hinglish** (gaps G1, G2 and G8). Part 3 builds that test set from **real public bank transactions** and runs Part 1/2's winning methods on it.

**Data (real, public, no synthetic rows):**

| Ledger | What it is | Why |
|---|---|---|
| **Indian bank statements** (Kaggle `apoorvwatsky/bank-transaction-data`) | Real extracted statements of several Indian bank accounts, INR, with narration, withdrawal, deposit and running balance | Indian narrations (NEFT, RTGS, IMPS, UPI-style text) and Indian amounts |
| **MoneyData** (Firat et al., EuroVis 2023; Mendeley `10.17632/dnxtg6n4rv`, CC BY 4.0) | One real person's anonymised retail current account over 7 years (6,500+ transactions), UK | A genuinely *personal* ledger, so results are not only about business accounts |

**How the test set is built.** The ledger is cut into monthly *statements* (what a user would upload). For each statement, questions are filled in from fixed templates written once in **English** and in **Hinglish**, and every gold answer is **computed by code** from the real rows with exact decimal arithmetic. Every English question has a Hinglish twin over the same rows with the same gold answer, so the language gap is a paired comparison. The question wording is templated; a second slice written by real users (the 10–15 person check) can be added later through `EXTRA_QUESTIONS_PATH`.

| Exp | Question | Arms |
|---|---|---|
| **E6** | Which method answers questions about a person's own statement correctly, in English and Hinglish, and when can the answer be marked *verified*? | **S1** direct answer · **S2** single agent + calculator (Part 2's best) · **P1** reason-then-program, numbers copied from the statement (Part 1's best) · **T1** code over the ledger: the program reads the rows itself (FinSight's engine design) · **Verified badge**: T1 kept only when another arm agrees, shown as an accuracy-vs-coverage curve |
| **E7** | Does transcript memory (Part 2's winner) still help on follow-up questions about a person's own spending, in both languages? | No memory · Transcript |

Scoring is strict, because these are a person's own money: an amount is right only within **0.50** (rupee or pound) of the gold, a count must be exact, and a percentage within 0.1 points. Part 1's looser 1% rule is reported too, for comparison.

**This notebook runs only the API model (no GPU needed). Settings: Accelerator None, Internet on, and a Kaggle secret named `LLM_API_KEY`** (Add-ons → Secrets). It builds the same test set with the same seed, and answers the first `API_E6_N` questions per ledger, so its results pair with the local run.

## 0. Configuration

**Before running:** in the right-hand panel, click **Add Input** and add the Kaggle dataset **"Bank Transaction Data" by apoorvwatsky**. MoneyData is downloaded from Mendeley automatically (turn **Internet on** in Settings). If the download fails, download `laramee26openBankTransactionData.xlsx` from https://data.mendeley.com/datasets/dnxtg6n4rv/1, upload it as a private Kaggle dataset, and add it as an input: the loader finds it under `/kaggle/input`.

In [ ]:
FAST = False          # True: small samples for a quick end-to-end check
SEED = 13

RUN_E6_LEDGER_QA = True
RUN_E7_MEMORY    = True

# Ledgers to use. Each is found under /kaggle/input (or DATA_DIRS), or downloaded.
DATASETS = {
    "indian_bank": dict(kaggle_slug="apoorvwatsky/bank-transaction-data", name_hints=["bank"],
                        currency="INR", dayfirst=True, columns={}),
    "moneydata":   dict(mendeley_id="dnxtg6n4rv", mendeley_version=1, name_hints=["laramee", "openbank", "moneydata"],
                        currency="GBP", dayfirst=True, columns={}),
}
# If the loader picks the wrong column, name it here, e.g. columns={"date": "DATE", "debit": " WITHDRAWAL AMT "}.
DATA_DIRS = ["/kaggle/input"]
EXTRA_QUESTIONS_PATH = None   # optional JSONL of user-written questions (see README), added to E6

# Local backend (runs when USE_LOCAL is True and a GPU is present)
USE_LOCAL    = False
LLM_NAME     = "Qwen/Qwen2.5-7B-Instruct"   # loaded in 4-bit; falls back to FALLBACK_LLM if 4-bit loading fails
LLM_4BIT     = True
FALLBACK_LLM = "Qwen/Qwen2.5-3B-Instruct"

# API backend (runs only if a key is found in Kaggle Secrets under API_SECRET_NAME)
USE_API         = True
RESULTS_NAME    = "part3_api_results"   # output folder name under /kaggle/working
API_SECRET_NAME = "LLM_API_KEY"
API_BASE_URL    = "https://generativelanguage.googleapis.com/v1beta/openai/"   # Groq: https://api.groq.com/openai/v1/
API_MODEL       = "gemini-3.5-flash-lite"   # free tier. Groq (free): openai/gpt-oss-20b. The notebook prints the models your key can use.
API_RPM         = 10      # requests per minute; keep under your free-tier limit
API_REASONING_EFFORT = "low"
API_EXTRA_TOKENS     = 1024

# Test set size
STATEMENT_MAX_ROWS   = 80     # rows shown in one statement (a month is cut at a day boundary if longer)
MONTH_MIN_ROWS       = 6      # months with fewer rows are skipped
STATEMENTS_PER_LEDGER = 25 if FAST else 120
QUESTIONS_PER_STATEMENT = 3
MAX_STATEMENTS_PER_ACCOUNT = 40   # so one busy account does not dominate
CONVS_PER_LEDGER     = 5 if FAST else 30    # E7 conversations per ledger (each in EN and Hinglish)

# How many items each backend answers (the API takes the first API_* of the same sample, so results pair up)
E6_N_PER_LEDGER  = 30 if FAST else 150   # per ledger, per language
API_E6_N         = 10 if FAST else 40
API_CONVS        = 3 if FAST else 12

AGENT_MAX_STEPS = 3
GEN_BATCH       = 16
USE_ALL_GPUS    = False
TOKEN_BUDGET    = 12000

## 1. Setup

In [ ]:
import subprocess, sys, importlib

def ensure(pkg, pip_name=None):
    try:
        importlib.import_module(pkg)
    except Exception:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or pkg], check=False)

for pkg, pip_name in [("transformers", None), ("openpyxl", None), ("scipy", None), ("pandas", None),
                      ("matplotlib", None), ("requests", None)]:
    ensure(pkg, pip_name)
if LLM_4BIT and USE_LOCAL:
    ensure("bitsandbytes")

In [ ]:
import os, re, json, math, time, random, zipfile, hashlib, signal, urllib.request, platform, glob, datetime as dt
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
from collections import Counter, defaultdict
from decimal import Decimal, ROUND_HALF_UP, InvalidOperation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

random.seed(SEED); np.random.seed(SEED)

OUT = os.path.join("/kaggle/working", RESULTS_NAME) if os.path.isdir("/kaggle/working") else os.path.abspath(RESULTS_NAME)
CACHE = os.path.join(OUT, "cache")
os.makedirs(CACHE, exist_ok=True)

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(SEED)
except ImportError:
    torch, DEVICE = None, "cpu"

def dtype_kw(dtype):
    import transformers
    major, minor = (int(x) for x in re.findall(r"\d+", transformers.__version__)[:2])
    return {"dtype" if (major, minor) >= (4, 56) else "torch_dtype": dtype}

RESULTS = {"config": dict(FAST=FAST, SEED=SEED, LLM_NAME=LLM_NAME, LLM_4BIT=LLM_4BIT, API_BASE_URL=API_BASE_URL,
                          API_MODEL=API_MODEL, STATEMENT_MAX_ROWS=STATEMENT_MAX_ROWS, E6_N_PER_LEDGER=E6_N_PER_LEDGER,
                          API_E6_N=API_E6_N, CONVS_PER_LEDGER=CONVS_PER_LEDGER, API_CONVS=API_CONVS,
                          AGENT_MAX_STEPS=AGENT_MAX_STEPS),
           "env": dict(python=platform.python_version(), device=DEVICE,
                       gpu=(torch.cuda.get_device_name(0) if torch is not None and DEVICE == "cuda" else None),
                       torch=getattr(torch, "__version__", None)),
           "data": {}, "E6": {}, "E7": {}}
SUMMARY_ROWS = []
print(RESULTS["env"], "| output:", OUT)

def download(url, name):
    path = os.path.join(CACHE, name)
    if not os.path.exists(path):
        print("downloading", url)
        req = urllib.request.Request(url, headers={"User-Agent": "finsight-research"})
        with urllib.request.urlopen(req, timeout=300) as r, open(path, "wb") as f:
            f.write(r.read())
    return path

def bootstrap_ci(values, n=1000, seed=SEED):
    values = np.asarray(values, dtype=float)
    if len(values) == 0:
        return (float("nan"), float("nan"))
    rng = np.random.default_rng(seed)
    means = values[rng.integers(0, len(values), size=(n, len(values)))].mean(axis=1)
    return (float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5)))

def paired_diff_ci(a, b, n=1000, seed=SEED):
    d = np.asarray(a, dtype=float) - np.asarray(b, dtype=float)
    if len(d) == 0:
        return float("nan"), float("nan"), float("nan")
    lo, hi = bootstrap_ci(d, n, seed)
    return float(d.mean()), lo, hi

def add_summary(exp, backend, method, metric, value, ci=None, **extra):
    row = dict(experiment=exp, backend=backend, method=method, metric=metric, value=round(float(value), 4))
    if ci is not None:
        row["ci_low"], row["ci_high"] = round(ci[0], 4), round(ci[1], 4)
    row.update({k: (round(float(v), 4) if isinstance(v, (float, np.floating)) else v) for k, v in extra.items()})
    SUMMARY_ROWS.append(row)

def save_all():
    pd.DataFrame(SUMMARY_ROWS).to_csv(os.path.join(OUT, "summary.csv"), index=False)
    with open(os.path.join(OUT, "results.json"), "w") as f:
        json.dump(RESULTS, f, indent=2, default=str)

## 2. Load the real ledgers

The loader finds each file, guesses the columns from their names (date, description, debit, credit or signed amount, balance, account) and prints what it chose, so a wrong guess is visible. Rows keep the file's own order within each day, which is the order a statement shows. It also checks how often the running balance agrees with the amounts (`balance[i-1] - debit + credit = balance[i]`), as a data-quality figure for the paper.

In [ ]:
COLUMN_HINTS = {
    "account":   ["accountno", "accountnumber", "account", "acno"],
    "date":      ["date", "transactiondate", "txndate", "postingdate", "valuedate"],
    "narration": ["transactiondetails", "transactiondescription", "description", "narration", "details",
                  "particulars", "memo", "reference"],
    "debit":     ["withdrawalamt", "withdrawal", "debitamount", "debit", "paidout", "moneyout"],
    "credit":    ["depositamt", "deposit", "creditamount", "credit", "paidin", "moneyin"],
    "amount":    ["amount", "transactionamount", "debitorcreditamount"],
    "balance":   ["balanceamt", "balance", "runningbalance", "closingbalance", "remainingbalance"],
    "type":      ["transactiontype", "type"],
}

def norm_col(c):
    return re.sub(r"[^a-z]", "", str(c).lower())

def guess_columns(cols, override):
    found, used = {}, set()
    normed = {c: norm_col(c) for c in cols}
    for role, hints in COLUMN_HINTS.items():
        if role in override:
            found[role] = override[role]; used.add(override[role]); continue
        pick = None
        for h in hints:                      # exact name first, in hint order
            pick = next((c for c in cols if c not in used and normed[c] == h), None)
            if pick is not None:
                break
        if pick is None:
            for h in hints:                  # then "contains"
                if len(h) < 4:
                    continue
                pick = next((c for c in cols if c not in used and h in normed[c]
                             and not (role == "date" and "value" in normed[c] and h == "date")), None)
                if pick is not None:
                    break
        if pick is not None:
            found[role] = pick; used.add(pick)
    return found

def to_decimal(x):
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return Decimal("0")
    s = str(x).strip().replace(",", "").replace("₹", "").replace("£", "").replace("$", "")
    s = re.sub(r"(?i)\b(inr|gbp|rs\.?|cr|dr)\b", "", s).strip()
    if s in {"", "-", ".", "nan", "None", "NaT"}:
        return Decimal("0")
    neg = s.startswith("(") and s.endswith(")")
    try:
        d = Decimal(s.strip("()"))
    except InvalidOperation:
        return Decimal("0")
    return -d if neg else d

def to_balance(x):
    if x is None or (isinstance(x, float) and math.isnan(x)) or str(x).strip() in {"", "nan", "None"}:
        return None
    return to_decimal(x)

def find_files(spec):
    files = []
    for root in DATA_DIRS:
        for ext in ("xlsx", "xls", "csv"):
            files += glob.glob(os.path.join(root, "**", f"*.{ext}"), recursive=True)
    hits = [f for f in files if any(h in os.path.basename(f).lower() or h in f.lower().replace("-", "")
                                    for h in spec["name_hints"])]
    if spec.get("kaggle_slug"):
        slug_dir = spec["kaggle_slug"].split("/")[-1]
        hits = [f for f in hits if slug_dir in f] or hits
    return sorted(set(hits))

def fetch(spec):
    files = find_files(spec)
    if spec.get("mendeley_id") and not any("laramee" in f.lower() or "moneydata" in f.lower() for f in files):
        files = []
    if files:
        return files
    if spec.get("kaggle_slug"):
        try:
            import kagglehub
            d = kagglehub.dataset_download(spec["kaggle_slug"])
            return sorted(glob.glob(os.path.join(d, "**", "*.xls*"), recursive=True) +
                          glob.glob(os.path.join(d, "**", "*.csv"), recursive=True))
        except Exception as exc:
            print(f"  kagglehub download failed ({type(exc).__name__}): add the dataset with Add Input")
    if spec.get("mendeley_id"):
        try:
            api = (f"https://data.mendeley.com/public-api/datasets/{spec['mendeley_id']}/files"
                   f"?folder_id=root&version={spec.get('mendeley_version', 1)}")
            out = []
            for f in requests.get(api, timeout=60).json():
                name = f["filename"]
                if name.lower().endswith((".xlsx", ".xls", ".csv")):
                    out.append(download(f["content_details"]["download_url"], name))
            return out
        except Exception as exc:
            print(f"  Mendeley download failed ({type(exc).__name__}: {exc}). Upload the file as a Kaggle input instead.")
    return []

def read_tables(path):
    if path.lower().endswith(".csv"):
        for enc in ("utf-8", "latin-1"):
            try:
                return [pd.read_csv(path, encoding=enc)]
            except UnicodeDecodeError:
                continue
    sheets = pd.read_excel(path, sheet_name=None)
    return list(sheets.values())

def header_fix(df):
    """Some statement exports have title rows above the header: use the first row that looks like a header."""
    if guess_columns(list(df.columns), {}).get("date") is not None:
        return df
    for i in range(min(15, len(df))):
        row = [str(v) for v in df.iloc[i].tolist()]
        if guess_columns(row, {}).get("date") is not None:
            new = df.iloc[i + 1:].copy()
            new.columns = row
            return new
    return df

DEBIT_TYPES = re.compile(r"(?i)deb|^dr$|withdraw|out|payment|^bp$|^c/p$|^chq$|^d/d$|^fee$|^fpo$|^so$|^pay$")

def load_ledger(key, spec):
    files = fetch(spec)
    if not files:
        print(f"[{key}] no file found: skipped")
        return None
    frames = []
    for path in files:
        for raw in read_tables(path):
            raw = header_fix(raw).dropna(how="all").reset_index(drop=True)
            raw.columns = [str(c) for c in raw.columns]
            cols = guess_columns(list(raw.columns), spec.get("columns", {}))
            if "date" not in cols or not ({"debit", "credit"} & set(cols) or "amount" in cols):
                continue
            print(f"[{key}] {os.path.basename(path)}: {len(raw)} rows, columns used: {cols}")
            d = pd.DataFrame(index=raw.index)
            d["account"] = raw[cols["account"]].astype(str).str.strip().str.strip("'") if "account" in cols else "A1"
            d["date"] = pd.to_datetime(raw[cols["date"]], dayfirst=spec.get("dayfirst", True), errors="coerce")
            d["narration"] = (raw[cols["narration"]].astype(str) if "narration" in cols else "").fillna("")
            d["narration"] = d["narration"].replace({"nan": ""}).str.replace(r"\s+", " ", regex=True).str.strip()
            if "type" in cols:
                tcol = raw[cols["type"]].astype(str).replace({"nan": ""}).str.strip()
                d["narration"] = np.where(tcol != "", tcol + " " + d["narration"], d["narration"])
            if {"debit", "credit"} <= set(cols):
                d["debit"] = [abs(to_decimal(v)) for v in raw[cols["debit"]]]
                d["credit"] = [abs(to_decimal(v)) for v in raw[cols["credit"]]]
            else:
                amt = [to_decimal(v) for v in raw[cols["amount"] if "amount" in cols else cols.get("debit", cols.get("credit"))]]
                if any(a < 0 for a in amt):
                    d["debit"] = [-a if a < 0 else Decimal("0") for a in amt]
                    d["credit"] = [a if a > 0 else Decimal("0") for a in amt]
                elif "type" in cols:
                    is_deb = [bool(DEBIT_TYPES.search(str(t).strip())) for t in raw[cols["type"]]]
                    d["debit"] = [a if dbt else Decimal("0") for a, dbt in zip(amt, is_deb)]
                    d["credit"] = [Decimal("0") if dbt else a for a, dbt in zip(amt, is_deb)]
                    print(f"[{key}] amounts are unsigned: debit/credit taken from the transaction type column")
                else:
                    print(f"[{key}] cannot tell debits from credits: set columns= in DATASETS"); continue
            d["balance"] = [to_balance(v) for v in raw[cols["balance"]]] if "balance" in cols else None
            d["file_order"] = np.arange(len(d))
            frames.append(d)
    if not frames:
        print(f"[{key}] no usable table found: skipped")
        return None
    df = pd.concat(frames, ignore_index=True)
    df = df[df.date.notna() & ((df.debit > 0) | (df.credit > 0))].copy()
    df["date"] = df.date.dt.date
    df = df.sort_values(["account", "date", "file_order"], kind="mergesort").reset_index(drop=True)

    # running-balance check, within each account in statement order
    ok = tot = 0
    for _, g in df.groupby("account", sort=False):
        b = list(g.balance)
        for i in range(1, len(g)):
            if b[i] is None or b[i - 1] is None:
                continue
            tot += 1
            ok += abs(b[i - 1] - g.debit.iloc[i] + g.credit.iloc[i] - b[i]) <= Decimal("0.01")
    has_bal = df.balance.notna().any() if "balance" in df else False
    info = dict(rows=len(df), accounts=df.account.nunique(), first=str(df.date.min()), last=str(df.date.max()),
                currency=spec["currency"], balance_column=bool(has_bal),
                balance_consistent=(round(ok / tot, 4) if tot else None),
                debits=int((df.debit > 0).sum()), credits=int((df.credit > 0).sum()))
    print(f"[{key}] {info}")
    RESULTS["data"][key] = info
    df["has_balance"] = bool(has_bal) and (info["balance_consistent"] or 0) >= 0.9
    return df

LEDGERS = {}
for key, spec in DATASETS.items():
    led = load_ledger(key, spec)
    if led is not None and len(led):
        LEDGERS[key] = led
if not LEDGERS:
    raise RuntimeError("No ledger loaded. Add the Kaggle dataset with Add Input, or check the messages above.")
for key, led in LEDGERS.items():
    print(f"\n[{key}] first rows:")
    print(led.head(5).to_string())

## 3. Build the test set

**Statements.** Each account is cut into calendar months. A month with more than `STATEMENT_MAX_ROWS` rows is cut at the last whole day that fits, and its period says so ("1 to 12 March 2018"). Some statements cover two whole consecutive months, so month-to-month questions are possible.

**Question types** (each with two English and two Hinglish wordings): money out, money in, net flow, number of debits, largest debit, average debit, spending on a word in the description, balance at the end of a day, change from one month to the next, and percentage change. Gold answers use `Decimal`, so there is no float rounding in the gold.

`ledger_qa_items.jsonl` in the output folder is the test set itself: statement rows, questions in both languages, gold answers and type.

In [ ]:
MONTHS = ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October",
          "November", "December"]

def d_long(d):
    return f"{d.day} {MONTHS[d.month - 1]} {d.year}"

def d_short(d):
    return f"{d.day:02d}-{MONTHS[d.month - 1][:3]}-{d.year}"

def period_text(chunk, lang):
    s, e = chunk["start"], chunk["end"]
    m = f"{MONTHS[s.month - 1]} {s.year}"
    if chunk["full_month"]:
        return m
    return f"1 to {e.day} {m}" if lang == "en" else f"1 se {e.day} {m}"

def month_chunks(g):
    out = []
    for (y, m), mg in g.groupby([g.date.map(lambda d: d.year), g.date.map(lambda d: d.month)], sort=True):
        if len(mg) < MONTH_MIN_ROWS:
            continue
        start = dt.date(y, m, 1)
        nxt = dt.date(y + (m == 12), m % 12 + 1, 1)
        end = nxt - dt.timedelta(days=1)
        full = True
        if len(mg) > STATEMENT_MAX_ROWS:
            counts = mg.groupby("date", sort=True).size()
            keep_days, n = [], 0
            for day, c in counts.items():
                if n + c > STATEMENT_MAX_ROWS:
                    break
                keep_days.append(day); n += c
            if n < MONTH_MIN_ROWS:
                continue
            mg = mg[mg.date.isin(keep_days)]
            end, full = keep_days[-1], False
        out.append(dict(start=start, end=end, full_month=full, rows=mg))
    return out

def build_statements(key, led, rng):
    stmts = []
    for acct, g in led.groupby("account", sort=True):
        chunks = month_chunks(g)
        acct_stmts = []
        i = 0
        while i < len(chunks):
            c = chunks[i]
            nxt = chunks[i + 1] if i + 1 < len(chunks) else None
            pair_ok = (nxt is not None and c["full_month"] and nxt["full_month"]
                       and (nxt["start"] - c["end"]).days == 1
                       and len(c["rows"]) + len(nxt["rows"]) <= STATEMENT_MAX_ROWS)
            if pair_ok and rng.random() < 0.5:
                acct_stmts.append([c, nxt]); i += 2
            else:
                acct_stmts.append([c]); i += 1
        rng.shuffle(acct_stmts)
        for ch in acct_stmts[:MAX_STATEMENTS_PER_ACCOUNT]:
            stmts.append(dict(ledger=key, account=acct, chunks=ch))
    rng.shuffle(stmts)
    stmts = stmts[:STATEMENTS_PER_LEDGER]
    for j, s in enumerate(stmts):
        s["id"] = f"{key}-{j:04d}"
        s["rows"] = pd.concat([c["rows"] for c in s["chunks"]])
        s["has_balance"] = bool(s["rows"].has_balance.iloc[0]) and s["rows"].balance.notna().all()
    return stmts

def fmt_amt(x):
    return "" if x is None or x == 0 else f"{x:.2f}"

def statement_text(s):
    cur = DATASETS[s["ledger"]]["currency"]
    per_en = " and ".join(period_text(c, "en") for c in s["chunks"])
    head = f"Bank statement, account {s['account']} ({cur}). Period: {per_en}."
    cols = "Date | Description | Debit | Credit" + (" | Balance" if s["has_balance"] else "")
    lines = [head, cols]
    for r in s["rows"].itertuples():
        row = f"{d_short(r.date)} | {r.narration[:60]} | {fmt_amt(r.debit)} | {fmt_amt(r.credit)}"
        if s["has_balance"]:
            row += f" | {r.balance:.2f}"
        lines.append(row)
    return "\n".join(lines)

def statement_records(s):
    return [dict(date=r.date.isoformat(), description=r.narration[:60], debit=float(r.debit), credit=float(r.credit),
                 balance=(float(r.balance) if s["has_balance"] else None)) for r in s["rows"].itertuples()]

TEMPLATES = {
    "total_out": dict(
        en=["How much money went out of my account in {p}?", "What were my total debits in {p}?"],
        hi=["{p} mein mere account se total kitna paisa gaya?", "{p} mein kul kitne ke debits hue?"]),
    "total_in": dict(
        en=["How much money came into my account in {p}?", "What were my total credits in {p}?"],
        hi=["{p} mein mere account mein total kitna paisa aaya?", "{p} mein kul kitna credit hua?"]),
    "net_flow": dict(
        en=["What was my net cash flow in {p} (money in minus money out)?",
            "In {p}, how much more came in than went out? Give a negative number if more went out."],
        hi=["{p} mein net kitna bacha? (jo aaya minus jo gaya, zyada gaya ho to minus mein batao)",
            "{p} ka net cash flow kya tha, credits minus debits?"]),
    "count_out": dict(
        en=["How many debit transactions were there in {p}?", "How many times did money go out of my account in {p}?"],
        hi=["{p} mein kitne debit transactions hue?", "{p} mein kitni baar mere account se paisa gaya?"]),
    "max_out": dict(
        en=["What was my largest single debit in {p}?", "What was the biggest amount that went out in one transaction in {p}?"],
        hi=["{p} mein sabse bada single debit kitne ka tha?", "{p} mein ek transaction mein sabse zyada kitna paisa gaya?"]),
    "avg_out": dict(
        en=["What was the average debit amount in {p}?", "On average, how much was each debit in {p}?"],
        hi=["{p} mein average debit kitne ka tha?", "{p} mein har debit transaction average kitne ka raha?"]),
    "keyword_out": dict(
        en=["How much went out in {p} in transactions whose description contains \"{kw}\"?",
            "What did I pay in total in {p} for debits that mention \"{kw}\" in the description?"],
        hi=["{p} mein jin debits ke description mein \"{kw}\" likha hai, unpe total kitna paisa gaya?",
            "{p} mein \"{kw}\" wale debits ka total kitna hai?"]),
    "balance_eod": dict(
        en=["What was my account balance at the end of {d}?", "How much was left in my account after the last transaction on {d}?"],
        hi=["{d} ko din ke end mein mera balance kitna tha?", "{d} ki last entry ke baad account mein kitna balance tha?"]),
    "month_diff": dict(
        en=["How much more did I spend in {m1} than in {m2}? Give a negative number if I spent less.",
            "Total debits in {m1} minus total debits in {m2}: what is the difference?"],
        hi=["{m1} mein {m2} se kitna zyada kharch hua? Kam hua ho to minus mein batao.",
            "{m1} ke total debits minus {m2} ke total debits kitna hai?"]),
    "month_pct": dict(
        en=["By what percentage did my total debits change from {m1} to {m2}?",
            "What was the percentage change in money going out from {m1} to {m2}?"],
        hi=["{m1} se {m2} tak mere total debits kitne percent badle?",
            "{m1} ki tulna mein {m2} mein kharcha kitne percent badla?"]),
}
ANSWER_KIND = {"count_out": "count", "month_pct": "percent"}

STOPWORDS = {"FROM", "WITH", "THE", "AND", "FOR", "INTO", "TRANSFER", "PAYMENT", "BANK", "ACCOUNT", "NONE", "NULL"}

def q2(x):
    return x.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)

def gold_for(qtype, rows, extra):
    deb = [r for r in rows.itertuples() if r.debit > 0]
    if qtype == "total_out":
        return sum((r.debit for r in deb), Decimal("0"))
    if qtype == "total_in":
        return sum((r.credit for r in rows.itertuples()), Decimal("0"))
    if qtype == "net_flow":
        return sum((r.credit - r.debit for r in rows.itertuples()), Decimal("0"))
    if qtype == "count_out":
        return Decimal(len(deb))
    if qtype == "max_out":
        return max(r.debit for r in deb)
    if qtype == "avg_out":
        return q2(sum((r.debit for r in deb), Decimal("0")) / len(deb))
    if qtype == "keyword_out":
        kw = extra["kw"].lower()
        return sum((r.debit for r in deb if kw in r.narration.lower()), Decimal("0"))
    if qtype == "balance_eod":
        day = rows[rows.date == extra["day"]]
        return day.balance.iloc[-1]
    raise ValueError(qtype)

def pick_keyword(rows, rng):
    deb = rows[rows.debit > 0]
    if len(deb) < 3:
        return None
    toks = Counter()
    for n in deb.narration:
        toks.update({t.upper() for t in re.findall(r"[A-Za-z]{3,}", n)} - STOPWORDS)
    cands = []
    for t, c in toks.items():
        sub = sum(t.lower() in n.lower() for n in deb.narration)
        if 2 <= c <= 0.7 * len(deb) and sub == c:   # a word in 2+ debits, not in nearly all of them
            cands.append(t)
    return rng.choice(sorted(cands)) if cands else None

def debits_total(rows):
    return sum((d for d in rows.debit), Decimal("0"))

def make_questions(s, rng):
    out = []
    rows = s["rows"]
    cand = []
    whole = s["chunks"][0] if len(s["chunks"]) == 1 else None
    # single-period questions are asked about one chunk (the whole statement when it has one)
    for ci, c in enumerate(s["chunks"]):
        r = c["rows"]
        base = dict(chunk=ci)
        for t in ("total_out", "total_in", "net_flow"):
            cand.append((t, base))
        if (r.debit > 0).sum() >= 1:
            cand += [("count_out", base), ("max_out", base), ("avg_out", base)]
        kw = pick_keyword(r, rng)
        if kw:
            cand.append(("keyword_out", dict(base, kw=kw)))
    if s["has_balance"]:
        day = rng.choice(sorted(set(rows.date)))
        cand.append(("balance_eod", dict(day=day)))
    if len(s["chunks"]) == 2:
        cand.append(("month_diff", {}))
        if debits_total(s["chunks"][0]["rows"]) > 0:
            cand.append(("month_pct", {}))
    rng.shuffle(cand)
    seen = set()
    for qtype, extra in cand:
        if qtype in seen:
            continue
        seen.add(qtype)
        if qtype in ("month_diff", "month_pct"):
            a, b = (debits_total(c["rows"]) for c in s["chunks"])
            gold = a - b if qtype == "month_diff" else q2((b - a) / a * 100)
            fill = {lang: dict(m1=period_text(s["chunks"][0], lang), m2=period_text(s["chunks"][1], lang))
                    for lang in ("en", "hi")}
        elif qtype == "balance_eod":
            gold = gold_for(qtype, rows, extra)
            fill = {lang: dict(d=d_long(extra["day"])) for lang in ("en", "hi")}
        else:
            c = s["chunks"][extra["chunk"]]
            gold = gold_for(qtype, c["rows"], extra)
            fill = {lang: dict(p=period_text(c, lang), kw=extra.get("kw", "")) for lang in ("en", "hi")}
        k = rng.randrange(2)
        out.append(dict(qtype=qtype, kind=ANSWER_KIND.get(qtype, "money"), gold=str(gold), wording=k,
                        question={lang: TEMPLATES[qtype][lang][k].format(**fill[lang]) for lang in ("en", "hi")},
                        extra={kk: str(v) for kk, v in extra.items()}))
        if len(out) >= QUESTIONS_PER_STATEMENT:
            break
    return out

rng = random.Random(SEED)
STATEMENTS, ITEMS = {}, []
for key, led in LEDGERS.items():
    for s in build_statements(key, led, rng):
        STATEMENTS[s["id"]] = s
        for qi, q in enumerate(make_questions(s, rng)):
            ITEMS.append(dict(id=f"{s['id']}-q{qi}", statement=s["id"], ledger=key, n_rows=len(s["rows"]), **q))

if EXTRA_QUESTIONS_PATH and os.path.exists(EXTRA_QUESTIONS_PATH):
    # Each line: {"statement": "<id>", "lang": "en"|"hi", "question": "...", "gold": "123.45", "kind": "money"}
    with open(EXTRA_QUESTIONS_PATH) as f:
        for j, line in enumerate(f):
            e = json.loads(line)
            if e["statement"] in STATEMENTS:
                ITEMS.append(dict(id=f"user-{j:04d}", statement=e["statement"], ledger=STATEMENTS[e["statement"]]["ledger"],
                                  n_rows=len(STATEMENTS[e["statement"]]["rows"]), qtype="user_written",
                                  kind=e.get("kind", "money"), gold=str(e["gold"]), wording=-1,
                                  question={e["lang"]: e["question"]}, extra={}))

with open(os.path.join(OUT, "ledger_qa_items.jsonl"), "w") as f:
    for it in ITEMS:
        s = STATEMENTS[it["statement"]]
        f.write(json.dumps(dict(it, statement_text=statement_text(s), account=s["account"]), default=str) + "\n")

RESULTS["data"]["test_set"] = dict(
    statements=len(STATEMENTS), items=len(ITEMS),
    by_ledger=dict(Counter(it["ledger"] for it in ITEMS)), by_type=dict(Counter(it["qtype"] for it in ITEMS)),
    rows_per_statement=dict(mean=float(np.mean([len(s["rows"]) for s in STATEMENTS.values()])),
                            max=int(max(len(s["rows"]) for s in STATEMENTS.values()))))
print(json.dumps(RESULTS["data"]["test_set"], indent=1))
for it in ITEMS[:4]:
    print(f"\n{it['id']} [{it['qtype']}] gold={it['gold']}\n  EN: {it['question'].get('en')}\n  HI: {it['question'].get('hi')}")

## 4. Shared tools

The program runner and the calculator are Part 1/2's. Answers are read from the last `Answer:` line; the parser accepts ₹, Rs, INR, £, Indian digit grouping (1,23,456) and "lakh"/"crore", so a correct answer in Indian style is not marked wrong.

In [ ]:
class _Timeout(Exception):
    pass

SAFE_BUILTINS = {"abs": abs, "min": min, "max": max, "sum": sum, "round": round, "len": len, "float": float, "int": int,
                 "str": str, "range": range, "pow": pow, "sorted": sorted, "bool": bool, "list": list, "dict": dict,
                 "set": set, "tuple": tuple, "zip": zip, "enumerate": enumerate, "any": any, "all": all,
                 "filter": filter, "map": map, "isinstance": isinstance, "reversed": reversed}

def run_program(code, seconds=5, variables=None):
    env = {"__builtins__": SAFE_BUILTINS, "math": math}
    env.update(variables or {})
    code = re.sub(r"^\s*(import|from)\s+(math|datetime)\b.*$", "", code, flags=re.M)
    if re.search(r"\b(import|open|exec|eval|globals|locals|getattr|setattr|__)", code):
        return None, "blocked: program uses a forbidden construct"
    def handler(signum, frame):
        raise _Timeout()
    old = signal.signal(signal.SIGALRM, handler) if hasattr(signal, "SIGALRM") else None
    try:
        if old is not None:
            signal.alarm(seconds)
        exec(code, env)
        if "ans" not in env:
            return None, "the program did not set a variable named ans"
        return env["ans"], None
    except _Timeout:
        return None, "the program timed out"
    except Exception as exc:
        return None, f"{type(exc).__name__}: {exc}"
    finally:
        if old is not None:
            signal.alarm(0); signal.signal(signal.SIGALRM, old)

def extract_code(text):
    m = re.search(r"```(?:python)?\s*\n(.*?)```", text, flags=re.S)
    if not m:
        m = re.search(r"```(?:python)?\s*\n(.*)", text, flags=re.S)
    code = m.group(1) if m else ""
    return "\n".join(l for l in code.splitlines() if not re.match(r"\s*answer\s*:", l, flags=re.I)).strip()

NUM_TOKEN = re.compile(r"-?\(?\d[\d,]*(?:\.\d+)?\)?")

def parse_amount(s):
    """'₹1,23,456.50', 'Rs. -2,000', '£45.10', '1.2 lakh', '(300.00)', '12.5%' -> float."""
    if s is None:
        return None
    s = str(s).replace("−", "-").replace("–", "-")
    s = re.sub(r"(?i)(₹|rs\.?|inr|£|gbp|\$|rupees?|pounds?)", " ", s)
    m = re.search(r"(-\s*)?\(?\d[\d,]*(?:\.\d+)?\)?", s)
    if not m:
        return None
    tok = m.group(0).replace(" ", "")
    neg = tok.startswith("-") or (tok.startswith("(") and tok.endswith(")"))
    try:
        v = float(tok.strip("-()").replace(",", ""))
    except ValueError:
        return None
    tail = s[m.end():m.end() + 12].lower()
    if re.match(r"\s*(lakh|lac)", tail):
        v *= 1e5
    elif re.match(r"\s*crore", tail):
        v *= 1e7
    elif re.match(r"\s*k\b", tail):
        v *= 1e3
    return -v if neg else v

def final_answer(text):
    lines = re.findall(r"(?im)^\W*(?:final\s+)?(?:answer|jawab|uttar)\s*\**\s*[:=\-]\s*\**(.+)$", text or "")
    if lines:
        return parse_amount(lines[-1])
    return None

def as_number(v):
    if isinstance(v, bool) or v is None:
        return None
    if isinstance(v, (int, float, Decimal, np.integer, np.floating)):
        f = float(v)
        return f if math.isfinite(f) else None
    if isinstance(v, str):
        return parse_amount(v)
    return None

def correct(pred, gold, kind, loose=False):
    if pred is None:
        return False
    g = float(gold)
    if loose:   # Part 1's FinQA rule: 1% relative, percent/fraction confusion accepted
        return any(abs(c - g) <= max(0.01 * abs(g), 5e-3) for c in (pred, pred * 100, pred / 100))
    if kind == "count":
        return abs(pred - g) < 1e-9
    if kind == "percent":
        return any(abs(c - g) <= 0.1 for c in (pred, pred * 100))
    return abs(pred - g) <= 0.5

def agree(a, b, kind):
    if a is None or b is None:
        return False
    if kind == "count":
        return abs(a - b) < 1e-9
    if kind == "percent":
        return any(abs(c - b) <= 0.1 for c in (a, a * 100, a / 100))
    return abs(a - b) <= 0.5

CALC_RE = re.compile(r"^\s*`?CALC\s*:\s*(.+?)`?\s*$", re.M | re.I)

def calc(expr):
    expr = expr.split("=")[0].strip().strip("`")
    expr = re.sub(r"(?i)(₹|rs\.?|£)", "", expr).replace("×", "*").replace("÷", "/")
    expr = re.sub(r"(?<=\d),(?=\d)", "", expr)
    val, err = run_program("ans = " + expr)
    if err:
        return f"error: {err}"
    return f"{val:.10g}" if isinstance(val, (int, float)) and not isinstance(val, bool) else str(val)

## 5. LLM backends

Same as Part 2: greedy decoding, identical prompts on both backends, token counts recorded, and every generation cached in `cache/generations.jsonl` so a re-run resumes where it stopped.

In [ ]:
SYSTEM = ("You are FinSight, a careful personal finance assistant. Answer questions about the user's own bank statement, "
          "using only the statement given. The user may write in English or in Hinglish (Hindi in Roman script).")

class APIStop(Exception):
    pass

class LocalBackend:
    name = "local"

    def __init__(self):
        self.replicas = []
        self._model_id = None

    @property
    def model_id(self):
        if self._model_id is None:
            self.load()
        return self._model_id

    def load(self):
        from transformers import AutoTokenizer, AutoModelForCausalLM
        def tokenizer(name):
            tok = AutoTokenizer.from_pretrained(name, padding_side="left", truncation_side="left")
            if tok.pad_token is None:
                tok.pad_token = tok.eos_token
            return tok
        def model_on(name, gpu, four_bit):
            kwargs = dict(device_map={"": gpu})
            if four_bit:
                from transformers import BitsAndBytesConfig
                kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
            else:
                kwargs.update(dtype_kw(torch.float16))
            return AutoModelForCausalLM.from_pretrained(name, **kwargs).eval()
        n_gpu = torch.cuda.device_count() if USE_ALL_GPUS else 1
        name, four_bit = LLM_NAME, LLM_4BIT
        try:
            self.replicas = [(tokenizer(name), model_on(name, 0, four_bit))]
        except Exception as exc:
            print(f"Could not load {LLM_NAME} ({type(exc).__name__}: {exc}); falling back to {FALLBACK_LLM} in fp16")
            name, four_bit = FALLBACK_LLM, False
            self.replicas = [(tokenizer(name), model_on(name, 0, four_bit))]
        for gpu in range(1, n_gpu):
            try:
                self.replicas.append((tokenizer(name), model_on(name, gpu, four_bit)))
            except Exception as exc:
                print(f"Could not load a copy on GPU {gpu} ({type(exc).__name__}); continuing with {len(self.replicas)} GPU(s)")
                break
        self._model_id = name + (" (4-bit)" if four_bit else "")
        print(f"Local LLM in use: {self._model_id} on {len(self.replicas)} GPU(s)")
        RESULTS["config"]["LOCAL_LLM_USED"] = self._model_id

    def run(self, convs, max_new_tokens):
        tok = self.replicas[0][0]
        texts = [tok.apply_chat_template([{"role": "system", "content": SYSTEM}] + c, tokenize=False,
                                         add_generation_prompt=True) for c in convs]
        lengths = [len(tok(t)["input_ids"]) for t in texts]
        batches, batch, longest = [], [], 0
        for t, n in zip(texts, lengths):
            longest_if_added = max(longest, n + max_new_tokens)
            if batch and (len(batch) >= GEN_BATCH or longest_if_added * (len(batch) + 1) > TOKEN_BUDGET):
                batches.append(batch)
                batch, longest_if_added = [], n + max_new_tokens
            batch.append(t); longest = longest_if_added
        if batch:
            batches.append(batch)
        if len(self.replicas) == 1 or len(batches) == 1:
            results = [self._generate(b, max_new_tokens, 0) for b in batches]
        else:
            from concurrent.futures import ThreadPoolExecutor
            results = [None] * len(batches)
            def worker(k):
                torch.cuda.set_device(k)
                for j in range(k, len(batches), len(self.replicas)):
                    results[j] = self._generate(batches[j], max_new_tokens, k)
            with ThreadPoolExecutor(len(self.replicas)) as pool:
                for f in [pool.submit(worker, k) for k in range(len(self.replicas))]:
                    f.result()
        return [rec for r in results for rec in r]

    def _generate(self, chunk, max_new_tokens, gpu):
        tok, model = self.replicas[gpu]
        t0 = time.perf_counter()
        try:
            enc = tok(chunk, return_tensors="pt", padding=True, truncation=True, max_length=8192).to(model.device)
            with torch.no_grad():
                gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id)
        except torch.cuda.OutOfMemoryError:
            enc = gen = None
            with torch.cuda.device(gpu):
                torch.cuda.empty_cache()
            if len(chunk) == 1:
                raise
            half = len(chunk) // 2
            return self._generate(chunk[:half], max_new_tokens, gpu) + self._generate(chunk[half:], max_new_tokens, gpu)
        new = gen[:, enc["input_ids"].shape[1]:]
        sec = (time.perf_counter() - t0) / len(chunk)
        decoded = tok.batch_decode(new, skip_special_tokens=True)
        out = [dict(text=text, in_tok=int(enc["attention_mask"][j].sum()),
                    out_tok=int((new[j] != tok.pad_token_id).sum()), sec=sec) for j, text in enumerate(decoded)]
        del enc, gen, new
        with torch.cuda.device(gpu):
            torch.cuda.empty_cache()
        return out

class APIBackend:
    name = "api"

    def __init__(self, key):
        self.key = key
        self.model_id = f"{API_MODEL} (effort {API_REASONING_EFFORT}, +{API_EXTRA_TOKENS} tokens)"
        self.url = API_BASE_URL.rstrip("/") + "/chat/completions"
        self.last, self.failures, self.effort, self.empty = 0.0, 0, API_REASONING_EFFORT, 0

    def run(self, convs, max_new_tokens):
        return [self._one(c, max_new_tokens) for c in convs]

    def _one(self, msgs, max_new_tokens):
        for attempt in range(6):
            body = {"model": API_MODEL, "temperature": 0, "max_tokens": max_new_tokens + API_EXTRA_TOKENS,
                    "messages": [{"role": "system", "content": SYSTEM}] + msgs}
            if self.effort:
                body["reasoning_effort"] = self.effort
            wait = self.last + 60.0 / API_RPM - time.time()
            if wait > 0:
                time.sleep(wait)
            self.last = time.time()
            t0 = time.perf_counter()
            try:
                r = requests.post(self.url, json=body, timeout=120,
                                  headers={"Authorization": f"Bearer {self.key}", "Content-Type": "application/json"})
            except requests.RequestException as exc:
                print("  API request failed:", type(exc).__name__)
                time.sleep(5 * 2 ** attempt)
                continue
            if r.status_code == 200:
                j = r.json()
                usage = j.get("usage") or {}
                self.failures = 0
                text = j["choices"][0]["message"].get("content") or ""
                if not text.strip():
                    self.empty += 1
                    if self.empty in (1, 10, 100):
                        print(f"  API returned an empty answer {self.empty} time(s); raise API_EXTRA_TOKENS if this keeps happening")
                return dict(text=text, in_tok=usage.get("prompt_tokens", 0), out_tok=usage.get("completion_tokens", 0),
                            sec=time.perf_counter() - t0, error=not text.strip())
            if r.status_code == 400 and self.effort and "reasoning" in r.text.lower():
                print(f"  this model does not accept reasoning_effort={self.effort!r}; sending requests without it")
                self.effort = None
                continue
            if r.status_code in (429, 500, 502, 503, 504):
                ra = r.headers.get("retry-after", "")
                time.sleep(min(float(ra) if re.fullmatch(r"\d+(\.\d+)?", ra) else 5 * 2 ** attempt, 120))
                continue
            raise APIStop(f"API error {r.status_code}: {r.text[:300]}")
        self.failures += 1
        if self.failures >= 3:
            raise APIStop("The API keeps refusing (rate limit or daily quota). Re-run later: finished answers are cached.")
        return dict(text="", in_tok=0, out_tok=0, sec=0.0, error=True)

GEN_STORE_PATH = os.path.join(CACHE, "generations.jsonl")
_GEN_STORE = {}
if os.path.exists(GEN_STORE_PATH):
    with open(GEN_STORE_PATH) as f:
        for line in f:
            rec = json.loads(line)
            _GEN_STORE[rec["key"]] = rec

def llm(backend, tag, convs, max_new_tokens):
    keys = [hashlib.sha1(json.dumps([backend.name, backend.model_id, max_new_tokens, SYSTEM, c]).encode()).hexdigest()
            for c in convs]
    todo = {k: c for k, c in zip(keys, convs) if k not in _GEN_STORE}
    print(f"[{backend.name}] {tag}: {len(convs) - len(todo)} cached, generating {len(todo)}", flush=True)
    order = sorted(todo, key=lambda k: len(json.dumps(todo[k])))
    step = GEN_BATCH * 8 if backend.name == "local" else 10
    t_start = time.perf_counter()
    for i in range(0, len(order), step):
        chunk = order[i:i + step]
        recs = backend.run([todo[k] for k in chunk], max_new_tokens)
        with open(GEN_STORE_PATH, "a") as f:
            for k, rec in zip(chunk, recs):
                if rec.get("error"):
                    continue
                rec["key"] = k
                _GEN_STORE[k] = rec
                f.write(json.dumps(rec) + "\n")
        print(f"  {min(i + step, len(order))}/{len(order)} ({time.perf_counter() - t_start:.0f}s)", end="\r", flush=True)
    if order:
        print()
    return [_GEN_STORE.get(k, dict(text="", in_tok=0, out_tok=0, sec=0.0, error=True)) for k in keys]

def get_api_key():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(API_SECRET_NAME)
    except Exception:
        return os.environ.get(API_SECRET_NAME)

BACKENDS = []
if USE_LOCAL and torch is not None and DEVICE == "cuda":
    BACKENDS.append(LocalBackend())
elif not USE_LOCAL:
    print("USE_LOCAL is False: the local backend is skipped.")
else:
    print("No GPU found: the local backend is skipped (turn on GPU T4 x2 in Settings).")
_key = get_api_key() if USE_API else None
if _key:
    BACKENDS.append(APIBackend(_key))
    print(f"API backend on: {API_MODEL} at {API_BASE_URL}")
    try:
        r = requests.get(API_BASE_URL.rstrip("/") + "/models", headers={"Authorization": f"Bearer {_key}"}, timeout=30)
        short = {m.get("id", "").split("models/")[-1] for m in r.json().get("data", [])}
        print("Models your key can use:", ", ".join(sorted(short)) or "(list not available)")
        if short and API_MODEL not in short:
            print(f"WARNING: {API_MODEL} is not in that list. Set API_MODEL to one of the names above and re-run this cell.")
    except Exception as exc:
        print("Could not list models:", type(exc).__name__)
elif USE_API:
    print(f"No secret named {API_SECRET_NAME}: the API backend is skipped.")

def user(text):
    return [{"role": "user", "content": text}]

## E6 · Questions about a person's own statement (English and Hinglish)

| Arm | What the model does | Where the numbers come from |
|---|---|---|
| **S1** · Direct | Reads the statement, thinks briefly, answers | The model's own arithmetic |
| **S2** · Agent + calculator | May call `CALC: <expression>` up to 3 times, then answers (Part 2's best single agent) | Numbers the model copies, arithmetic by the tool |
| **P1** · Reason, then program | Writes a Python program with the numbers it needs (Part 1's best on FinQA) | Numbers the model copies, arithmetic by code |
| **T1** · Code over the ledger | Writes a program over `txns`, the statement rows as data, and must not type amounts in | Rows read by code, arithmetic by code (FinSight's engine) |

**Verified badge.** T1's answer gets the badge when another arm reaches the same number by a different route. The result is reported as accuracy when verified, coverage, and a risk–coverage curve (how many other arms agree), which the literature review flagged as missing for consumer finance (gap G2).

In [ ]:
ANSWER_RULE = ("Give the final answer on its own last line as 'Answer: <number>', a plain number with no currency "
               "symbol and no commas. Money out and money in are positive amounts; use a minus sign only where the "
               "question asks for a difference that is negative. A percentage is given in percent (12.5 means 12.5%).")

def prompt_direct(it, lang, stmt):
    return (f"{stmt}\n\nQuestion: {it['question'][lang]}\n\n"
            f"Think step by step in a few short lines, then answer. {ANSWER_RULE}")

def prompt_agent(it, lang, stmt):
    return (f"{stmt}\n\nQuestion: {it['question'][lang]}\n\n"
            "You have a calculator. To use it, write one line 'CALC: <arithmetic expression>' and stop; you will get the "
            "result. You may use it up to 3 times. Copy amounts exactly from the statement. "
            f"When you are done, {ANSWER_RULE[0].lower() + ANSWER_RULE[1:]}")

PROGRAM_INSTR = ("First, in two or three short lines, say which amounts you need and the formula. Then write a short "
                 "Python program that computes the answer and stores it in a variable named ans. Copy every amount "
                 "exactly as it appears in the statement; do not invent or estimate values. Put the code in a ```python block.")

def prompt_program(it, lang, stmt):
    return f"{stmt}\n\nQuestion: {it['question'][lang]}\n\n{PROGRAM_INSTR}"

LEDGER_INSTR = ("The same statement rows are loaded in Python as `txns`, a list of dicts in statement order, with keys "
                "'date' (a datetime.date), 'description' (str), 'debit' (float, 0.0 if none), 'credit' (float, 0.0 if none) "
                "and 'balance' (float, or None if the statement has no balance). `date(y, m, d)` builds a date. "
                "Write a short Python program that computes the answer from `txns` and stores it in a variable named ans. "
                "Do not type amounts from the statement into the program: read them from `txns`. "
                "Explain the formula in one line first, then put the code in a ```python block.")

def prompt_ledger(it, lang, stmt):
    return f"{stmt}\n\nQuestion: {it['question'][lang]}\n\n{LEDGER_INSTR}"

def ledger_vars(s):
    recs = statement_records(s)
    for r in recs:
        r["date"] = dt.date.fromisoformat(r["date"])
    return {"txns": recs, "date": dt.date, "datetime": dt}

def typed_amounts(code, s):
    """Amounts from the statement that the model typed into a T1 program (it was told not to)."""
    amts = {f"{float(x):.2f}" for x in list(s["rows"].debit) + list(s["rows"].credit) if x > 0}
    nums = {f"{float(n):.2f}" for n in re.findall(r"(?<![\w.])\d+(?:\.\d+)?", code) if float(n) >= 10}
    return len(nums & amts)

def run_agent(backend, convs):
    records = [[] for _ in convs]
    final = [None] * len(convs)
    active = list(range(len(convs)))
    convs = [list(c) for c in convs]
    for step in range(AGENT_MAX_STEPS + 1):
        if not active:
            break
        if step == AGENT_MAX_STEPS:
            for i in active:
                convs[i] = convs[i] + user("Give your final answer now, as 'Answer: <number>'.")
        outs = llm(backend, f"S2 agent step {step + 1}", [convs[i] for i in active], 320)
        still = []
        for i, rec in zip(active, outs):
            records[i].append(rec)
            text = rec["text"]
            m = CALC_RE.search(text)
            ans_pos = re.search(r"(?im)^\W*answer\s*\**\s*[:=]", text)
            if m and step < AGENT_MAX_STEPS and (ans_pos is None or m.start() < ans_pos.start()):
                cut = text[:m.end()]
                convs[i] = convs[i] + [{"role": "assistant", "content": cut},
                                       {"role": "user", "content": f"Result: {calc(m.group(1))}"}]
                still.append(i)
            else:
                final[i] = text
        active = still
    return final, records

def tok_cost(recs):
    return sum(r.get("in_tok", 0) + r.get("out_tok", 0) for r in recs)

E6_ARMS = ["S1 direct", "S2 agent+calc", "P1 program (copied numbers)", "T1 code over ledger"]
E6_ROWS = []

def run_e6(backend, items):
    convs = {lang: {} for lang in ("en", "hi")}
    jobs = [(it, lang) for it in items for lang in ("en", "hi") if lang in it["question"]]
    stmt = {it["statement"]: statement_text(STATEMENTS[it["statement"]]) for it in items}
    P = {"S1 direct": prompt_direct, "P1 program (copied numbers)": prompt_program, "T1 code over ledger": prompt_ledger}
    outs = {}
    for arm, fn in P.items():
        recs = llm(backend, f"E6 {arm}", [user(fn(it, lang, stmt[it["statement"]])) for it, lang in jobs],
                   384 if arm == "S1 direct" else 512)
        outs[arm] = recs
    final, agent_recs = run_agent(backend, [user(prompt_agent(it, lang, stmt[it["statement"]])) for it, lang in jobs])
    rows = []
    for j, (it, lang) in enumerate(jobs):
        s = STATEMENTS[it["statement"]]
        row = dict(backend=backend.name, id=it["id"], ledger=it["ledger"], qtype=it["qtype"], kind=it["kind"], lang=lang,
                   gold=it["gold"], n_rows=it["n_rows"])
        preds, errors = {}, {}
        preds["S1 direct"] = final_answer(outs["S1 direct"][j]["text"])
        preds["S2 agent+calc"] = final_answer(final[j]) if final[j] else None
        for arm, variables in (("P1 program (copied numbers)", None), ("T1 code over ledger", ledger_vars(s))):
            code = extract_code(outs[arm][j]["text"])
            val, err = run_program(code, variables=variables) if code else (None, "no code block")
            preds[arm] = as_number(val)
            errors[arm] = err or ("not a number" if preds[arm] is None else None)
            if arm == "T1 code over ledger":
                row["t1_typed_amounts"] = typed_amounts(code, s) if code else 0
        costs = {"S1 direct": tok_cost([outs["S1 direct"][j]]), "S2 agent+calc": tok_cost(agent_recs[j]),
                 "P1 program (copied numbers)": tok_cost([outs["P1 program (copied numbers)"][j]]),
                 "T1 code over ledger": tok_cost([outs["T1 code over ledger"][j]])}
        for arm in E6_ARMS:
            row[f"pred::{arm}"] = preds[arm]
            row[f"ok::{arm}"] = correct(preds[arm], it["gold"], it["kind"])
            row[f"ok_loose::{arm}"] = correct(preds[arm], it["gold"], it["kind"], loose=True)
            row[f"tok::{arm}"] = costs[arm]
            if arm in errors:
                row[f"err::{arm}"] = errors[arm]
        t1 = preds["T1 code over ledger"]
        others = ["S1 direct", "S2 agent+calc", "P1 program (copied numbers)"]
        row["t1_agree_count"] = sum(agree(t1, preds[a], it["kind"]) for a in others)
        row["verified::T1+S1"] = agree(t1, preds["S1 direct"], it["kind"])
        row["verified::T1+P1"] = agree(t1, preds["P1 program (copied numbers)"], it["kind"])
        row["verified::T1+any"] = row["t1_agree_count"] >= 1
        rows.append(row)
    return rows

def e6_items_for(backend):
    out = []
    for key in LEDGERS:
        lst = [it for it in ITEMS if it["ledger"] == key and it["qtype"] != "user_written"]
        out += lst[:(API_E6_N if backend.name == "api" else E6_N_PER_LEDGER)]
    out += [it for it in ITEMS if it["qtype"] == "user_written"]
    return out

if RUN_E6_LEDGER_QA:
    for backend in BACKENDS:
        try:
            E6_ROWS += run_e6(backend, e6_items_for(backend))
        except APIStop as exc:
            print(f"[{backend.name}] stopped: {exc}")
    e6 = pd.DataFrame(E6_ROWS)
    if not e6.empty:
        e6.to_json(os.path.join(OUT, "e6_items.jsonl"), orient="records", lines=True)

In [ ]:
def summarise_e6(e6):
    tables = []
    for bk, g in e6.groupby("backend", sort=False):
        res = {}
        for arm in E6_ARMS:
            v = g[f"ok::{arm}"].astype(float).values
            ci = bootstrap_ci(v)
            res[arm] = dict(acc=float(v.mean()), ci=ci, loose=float(g[f"ok_loose::{arm}"].mean()),
                            tokens=float(g[f"tok::{arm}"].mean()))
            for lang in ("en", "hi"):
                res[arm][lang] = float(g[g.lang == lang][f"ok::{arm}"].mean()) if (g.lang == lang).any() else float("nan")
            # paired Hinglish - English gap on the same items
            pv = g.pivot_table(index="id", columns="lang", values=f"ok::{arm}", aggfunc="first").dropna()
            gap = paired_diff_ci(pv["hi"].astype(float), pv["en"].astype(float)) if {"en", "hi"} <= set(pv.columns) else (float("nan"),) * 3
            res[arm]["hi_minus_en"] = gap
            add_summary("E6 ledger QA", bk, arm, "accuracy (strict)", res[arm]["acc"], ci,
                        en=res[arm]["en"], hi=res[arm]["hi"], hi_minus_en=gap[0], gap_ci_low=gap[1], gap_ci_high=gap[2],
                        accuracy_loose=res[arm]["loose"], tokens=res[arm]["tokens"], n=len(g))
        # T1 versus each other arm, paired
        t1 = g["ok::T1 code over ledger"].astype(float).values
        for arm in E6_ARMS[:3]:
            d = paired_diff_ci(t1, g[f"ok::{arm}"].astype(float).values)
            res[arm]["t1_minus"] = d
        # verified badge
        badges = {}
        for b in ("verified::T1+S1", "verified::T1+P1", "verified::T1+any"):
            mask = g[b].astype(bool)
            acc_v = float(g[mask]["ok::T1 code over ledger"].mean()) if mask.any() else float("nan")
            badges[b] = dict(coverage=float(mask.mean()), accuracy_when_verified=acc_v,
                             accuracy_when_not=float(g[~mask]["ok::T1 code over ledger"].mean()) if (~mask).any() else float("nan"))
            add_summary("E6 verified badge", bk, b.split("::")[1], "accuracy when verified", acc_v if acc_v == acc_v else 0,
                        bootstrap_ci(g[mask]["ok::T1 code over ledger"].astype(float).values) if mask.any() else None,
                        coverage=badges[b]["coverage"], accuracy_when_not=badges[b]["accuracy_when_not"])
        curve = []
        for k in range(0, 4):
            mask = g.t1_agree_count >= k
            curve.append(dict(min_agreeing=k, coverage=float(mask.mean()),
                              accuracy=float(g[mask]["ok::T1 code over ledger"].mean()) if mask.any() else float("nan")))
        by_type = g.groupby("qtype")[[f"ok::{a}" for a in E6_ARMS]].mean().round(3)
        by_type.columns = E6_ARMS
        by_ledger = g.groupby(["ledger", "lang"])[[f"ok::{a}" for a in E6_ARMS]].mean().round(3)
        by_ledger.columns = E6_ARMS
        g = g.assign(rows_bucket=pd.cut(g.n_rows, [0, 20, 40, 60, 1000], labels=["<=20", "21-40", "41-60", ">60"]))
        by_len = g.groupby("rows_bucket", observed=True)[[f"ok::{a}" for a in E6_ARMS]].mean().round(3)
        by_len.columns = E6_ARMS
        err_counts = {a: dict(Counter(e for e in g.get(f"err::{a}", pd.Series(dtype=object)) if e)) for a in E6_ARMS[2:]}
        RESULTS["E6"][bk] = dict(arms=res, badges=badges, risk_coverage=curve, by_type=by_type.to_dict(),
                                 by_ledger={str(k): v for k, v in by_ledger.to_dict(orient="index").items()},
                                 by_statement_length=by_len.to_dict(),
                                 t1_programs_with_typed_amounts=float((g.t1_typed_amounts > 0).mean()),
                                 program_errors=err_counts, n=len(g))
        print(f"\n=== E6 · {bk} · {len(g)} answers ({g.id.nunique()} questions x languages) ===")
        print(pd.DataFrame({a: dict(accuracy=round(r["acc"], 3), ci=f"{r['ci'][0]:.3f}-{r['ci'][1]:.3f}",
                                    english=round(r["en"], 3), hinglish=round(r["hi"], 3),
                                    hinglish_minus_english=round(r["hi_minus_en"][0], 3),
                                    loose_1pct=round(r["loose"], 3), tokens=round(r["tokens"]))
                            for a, r in res.items()}).T.to_string())
        print("\nT1 minus other arm (paired, 95% CI):",
              {a: tuple(round(x, 3) for x in res[a]["t1_minus"]) for a in E6_ARMS[:3]})
        print("\nVerified badge on T1:", json.dumps(badges, indent=1))
        print("Risk-coverage (T1 kept when at least k other arms agree):", curve)
        print("\nBy question type:\n", by_type.to_string())
        print("\nBy ledger and language:\n", by_ledger.to_string())
        print("\nBy statement length (rows):\n", by_len.to_string())
        print("\nT1 programs that typed statement amounts in:", RESULTS["E6"][bk]["t1_programs_with_typed_amounts"])
        print("Program errors:", err_counts)
    save_all()

if RUN_E6_LEDGER_QA and E6_ROWS:
    summarise_e6(pd.DataFrame(E6_ROWS))

## E7 · Follow-up questions about one's own spending (memory)

Each conversation is four turns over a two-month statement, asked entirely in English or entirely in Hinglish:

1. How much did I spend in {month 1}? 2. And in {month 2}? 3. How much more did I spend in the first month than the second? 4. What is the total for both months together?

Turns 3 and 4 cannot be answered without remembering turns 1 and 2. Every turn uses T1 (code over the ledger), and the transcript holds the assistant's **own** earlier answers, never the gold ones. Arms: **No memory** (only the current question) and **Transcript** (Part 2's winner, +10.6 points on ConvFinQA).

In [ ]:
CONV_TEMPLATES = {
    "en": ["How much did I spend in {m1}?", "And in {m2}?",
           "How much more did I spend in the first month than in the second? Negative if less.",
           "What is the total for both months together?"],
    "hi": ["{m1} mein maine kitna kharch kiya?", "Aur {m2} mein?",
           "Pehle mahine mein doosre se kitna zyada kharch hua? Kam hua ho to minus mein batao.",
           "Dono mahino ka total kitna hua?"],
}
MEMORY_ARMS = ["No memory", "Transcript"]

def build_convs():
    convs = []
    for key in LEDGERS:
        two = [s for s in STATEMENTS.values() if s["ledger"] == key and len(s["chunks"]) == 2]
        for s in two[:CONVS_PER_LEDGER]:
            a, b = (debits_total(c["rows"]) for c in s["chunks"])
            golds = [a, b, a - b, a + b]
            for lang in ("en", "hi"):
                m1, m2 = (period_text(c, lang) for c in s["chunks"])
                qs = [t.format(m1=m1, m2=m2) for t in CONV_TEMPLATES[lang]]
                convs.append(dict(id=f"{s['id']}-{lang}", statement=s["id"], ledger=key, lang=lang, questions=qs,
                                  golds=[str(g) for g in golds]))
    return convs

def fmt_prev(a):
    return "unknown" if a is None else f"{a:.2f}"

def conv_prompt(arm, stmt, history, q):
    if arm == "Transcript" and history:
        convo = "".join(f"Q: {h['q']}\nA: {fmt_prev(h['a'])}\n" for h in history)
        return f"{stmt}\n\nConversation so far:\n{convo}\nCurrent question: {q}\n\n{LEDGER_INSTR}"
    return f"{stmt}\n\nQuestion: {q}\n\n{LEDGER_INSTR}"

def run_e7(backend, convs):
    rows = []
    for arm in MEMORY_ARMS:
        hist = {c["id"]: [] for c in convs}
        for t in range(4):
            prompts = []
            for c in convs:
                s = STATEMENTS[c["statement"]]
                prompts.append(user(conv_prompt(arm, statement_text(s), hist[c["id"]], c["questions"][t])))
            recs = llm(backend, f"E7 {arm} turn {t + 1}", prompts, 512)
            for c, rec in zip(convs, recs):
                s = STATEMENTS[c["statement"]]
                code = extract_code(rec["text"])
                val, err = run_program(code, variables=ledger_vars(s)) if code else (None, "no code block")
                pred = as_number(val)
                hist[c["id"]].append(dict(q=c["questions"][t], a=pred))
                rows.append(dict(backend=backend.name, arm=arm, conv=c["id"], ledger=c["ledger"], lang=c["lang"], turn=t + 1,
                                 gold=c["golds"][t], pred=pred, ok=correct(pred, c["golds"][t], "money"),
                                 in_tok=rec.get("in_tok", 0)))
    return rows

E7_ROWS = []
if RUN_E7_MEMORY:
    CONVS = build_convs()
    print(f"E7: {len(CONVS)} conversations ({len(CONVS) // 2} statements x 2 languages)")
    for backend in BACKENDS:
        n = len(CONVS) if backend.name == "local" else 2 * API_CONVS
        try:
            E7_ROWS += run_e7(backend, CONVS[:n])
        except APIStop as exc:
            print(f"[{backend.name}] stopped: {exc}")
    if E7_ROWS:
        e7 = pd.DataFrame(E7_ROWS)
        e7.to_json(os.path.join(OUT, "e7_items.jsonl"), orient="records", lines=True)
        for bk, g in e7.groupby("backend", sort=False):
            out = {}
            for arm, ga in g.groupby("arm", sort=False):
                follow = ga[ga.turn >= 3]
                whole = ga.groupby("conv").ok.all()
                out[arm] = dict(turn_acc=float(ga.ok.mean()), followup_acc=float(follow.ok.mean()),
                                conv_acc=float(whole.mean()), by_turn=ga.groupby("turn").ok.mean().round(3).to_dict(),
                                by_lang=ga.groupby("lang").ok.mean().round(3).to_dict(), prompt_tokens=float(ga.in_tok.mean()))
                add_summary("E7 memory", bk, arm, "follow-up accuracy (turns 3-4)", out[arm]["followup_acc"],
                            bootstrap_ci(follow.ok.astype(float).values), turn_accuracy=out[arm]["turn_acc"],
                            conversation_accuracy=out[arm]["conv_acc"], english=out[arm]["by_lang"].get("en"),
                            hinglish=out[arm]["by_lang"].get("hi"), prompt_tokens=out[arm]["prompt_tokens"])
            pv = g[g.turn >= 3].pivot_table(index=["conv", "turn"], columns="arm", values="ok", aggfunc="first").dropna()
            if set(MEMORY_ARMS) <= set(pv.columns):
                out["transcript_minus_none"] = paired_diff_ci(pv["Transcript"].astype(float), pv["No memory"].astype(float))
            RESULTS["E7"][bk] = out
            print(f"\n=== E7 · {bk} ===\n", json.dumps(out, indent=1, default=str))
        save_all()

## Summary

`summary.csv` has one row per backend and arm. For the paper: E6's main table (accuracy, English vs Hinglish, tokens), the T1-minus-other-arm paired differences, the verified-badge table and risk–coverage curve, and E7's follow-up accuracy. The results folder is zipped as `part3_results.zip` for upload.

In [ ]:
summary = pd.DataFrame(SUMMARY_ROWS)
save_all()
if not summary.empty:
    print(summary.to_string(index=False))
    fig, axes = plt.subplots(1, 3, figsize=(20, 5.5))
    colours = {"local": "#0c6b58", "api": "#b06a00"}
    ax = axes[0]
    part = summary[summary.experiment == "E6 ledger QA"]
    if not part.empty:
        y = np.arange(len(E6_ARMS)); h = 0.2
        k = 0
        for bk, g in part.groupby("backend", sort=False):
            g = g.set_index("method").reindex(E6_ARMS)
            for lang, shade in (("en", 1.0), ("hi", 0.55)):
                ax.barh(y + (k - 1.5) * h, g[lang], height=h, color=colours.get(bk, "#555"), alpha=shade,
                        label=f"{bk} · {'English' if lang == 'en' else 'Hinglish'}")
                k += 1
        ax.set_yticks(y, E6_ARMS, fontsize=8); ax.set_xlim(0, 1); ax.invert_yaxis()
        ax.set_title("E6 · strict accuracy on personal statements"); ax.legend(fontsize=7)
    ax = axes[1]
    for bk, r in RESULTS["E6"].items():
        c = pd.DataFrame(r["risk_coverage"])
        ax.plot(c.coverage, c.accuracy, marker="o", color=colours.get(bk, "#555"), label=bk)
        for _, p in c.iterrows():
            ax.annotate(f"≥{int(p.min_agreeing)}", (p.coverage, p.accuracy), fontsize=7, xytext=(3, 3), textcoords="offset points")
    ax.set_xlabel("coverage (share of questions answered)"); ax.set_ylabel("accuracy of T1 when answered")
    ax.set_xlim(0, 1.02); ax.set_ylim(0, 1.02); ax.set_title("E6 · verified badge: risk-coverage"); ax.legend()
    ax = axes[2]
    if E7_ROWS:
        e7 = pd.DataFrame(E7_ROWS)
        for (bk, arm), g in e7.groupby(["backend", "arm"], sort=False):
            t = g.groupby("turn").ok.mean()
            ax.plot(t.index, t.values, marker="o", linestyle="-" if arm == "Transcript" else "--",
                    color=colours.get(bk, "#555"), label=f"{bk} · {arm}")
        ax.set_xticks([1, 2, 3, 4]); ax.set_ylim(0, 1); ax.set_xlabel("turn"); ax.set_ylabel("accuracy")
        ax.set_title("E7 · follow-ups with and without memory"); ax.legend(fontsize=8)
    else:
        ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(os.path.join(OUT, "summary.png"), dpi=130)
    plt.show()

zip_path = os.path.join(os.path.dirname(OUT), f"{RESULTS_NAME}.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUT):
        for fn in files:
            if root.endswith("cache") and fn != "generations.jsonl":
                continue
            p = os.path.join(root, fn)
            z.write(p, os.path.relpath(p, os.path.dirname(OUT)))
print("Results zipped to", zip_path)